> **Note:** The the LangGraph agent is a mock so we don't need an OpenRouter key or a running Redmine instance to run most cells.

In [ ]:
print("Hello")

In [ ]:
!pip install fastapi pydantic langchain-core langgraph --quiet

In [ ]:
from fastapi import APIRouter, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field, model_validator
from typing import Dict, Any, Literal, Optional
import json
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.types import Command

print('APIRouter     → creates a group of related endpoints (like a blueprint)')
print('HTTPException → raises HTTP errors (e.g. 500) with a detail message')
print('StreamingResponse → sends data chunk by chunk (for SSE streaming)')
print('BaseModel     → Pydantic base class for request/response validation')
print('Field         → adds metadata/defaults to Pydantic fields')
print('model_validator → runs custom validation logic on the whole model')
print('Literal       → restricts a field to specific string values')
print('Command       → LangGraph instruction to resume a paused graph')

Pydantic models validate incoming request data and shape outgoing responses.

In [ ]:
class ChatRequest(BaseModel):
    message: str
    thread_id: str = 'default'

class ChatResponse(BaseModel):
    response: str
    requires_human: bool = False
    interrupts: Dict[str, Any] = Field(default_factory=dict)

# Test: valid request
req = ChatRequest(message='List open tasks', thread_id='session-1')
print('Valid ChatRequest:', req)

# Test: missing message field → should raise ValidationError
try:
    bad = ChatRequest(thread_id='abc')  # no message
except Exception as e:
    print('\nValidation error (expected):', e)

# Test: ChatResponse
resp = ChatResponse(response='Here are your tasks...', requires_human=False)
print('\nChatResponse:', resp.model_dump())

This model validates the body of the `/approve` endpoint.
The `@model_validator` runs after all fields are set and can raise errors.

In [ ]:
class ApproveRequest(BaseModel):
    decision_type: Literal['approve', 'reject', 'edit'] = 'approve'
    message: str = ''
    edited_action: Optional[Dict[str, Any]] = None

    @model_validator(mode='after')
    def validate_edit_payload(self):
        if self.decision_type == 'edit':
            if not self.edited_action:
                raise ValueError('edited_action is required when decision_type is edit')
            if 'name' not in self.edited_action or 'args' not in self.edited_action:
                raise ValueError('edited_action must contain name and args')
            if not isinstance(self.edited_action['args'], dict):
                raise ValueError('edited_action args must be a dict')
        return self

# Test 1: simple approve
a1 = ApproveRequest(decision_type='approve')
print('Approve:', a1.model_dump())

# Test 2: reject with message
a2 = ApproveRequest(decision_type='reject', message='Too risky')
print('Reject:', a2.model_dump())

# Test 3: valid edit
a3 = ApproveRequest(
    decision_type='edit',
    edited_action={'name': 'create_issue', 'args': {'subject': 'Updated title', 'priority_id': 4}}
)
print('Edit:', a3.model_dump())

# Test 4: invalid edit — missing edited_action
try:
    bad = ApproveRequest(decision_type='edit')
except Exception as e:
    print('\nValidation error (expected):', e)

# Test 5: invalid decision_type value
try:
    bad2 = ApproveRequest(decision_type='maybe')
except Exception as e:
    print('Literal error (expected):', e)

These two built-ins are used throughout the helper functions for safe introspection.

In [ ]:
# isinstance checks the TYPE of an object
x = {'key': 'value'}
print('isinstance(x, dict):', isinstance(x, dict))   # True
print('isinstance(x, str): ', isinstance(x, str))    # False
print('isinstance(x, list):', isinstance(x, list))   # False

# Why it matters: accessing .get() on a non-dict would crash
def safe_get(obj, key):
    if isinstance(obj, dict):
        return obj.get(key)
    return None

print('\nsafe_get on dict:', safe_get({'a': 1}, 'a'))
print('safe_get on str: ', safe_get('hello', 'a'))  # returns None instead of crashing

print()

# hasattr checks if an ATTRIBUTE exists on an object
class FakeResult:
    def __init__(self, has_interrupts):
        if has_interrupts:
            self.interrupts = ['interrupt_1']
        self.messages = []

r1 = FakeResult(has_interrupts=True)
r2 = FakeResult(has_interrupts=False)

print('hasattr(r1, interrupts):', hasattr(r1, 'interrupts'))  # True
print('hasattr(r2, interrupts):', hasattr(r2, 'interrupts'))  # False
print('hasattr(r1, messages):  ', hasattr(r1, 'messages'))    # True
print('hasattr(r1, value):     ', hasattr(r1, 'value'))       # False

# Why it matters: accessing r2.interrupts directly would crash
try:
    _ = r2.interrupts
except AttributeError as e:
    print('\nDirect access crash (expected):', e)

# Safe access with hasattr
if hasattr(r2, 'interrupts'):
    print('Has interrupts')
else:
    print('No interrupts (safe)')

Extracts interrupt objects from whatever `app.invoke()` returns.
Handles both new LangGraph API (`.interrupts`) and old API (`__interrupt__` key).

In [ ]:
def _extract_interrupts(result: Any) -> list:
    # New LangGraph API: result has a .interrupts attribute
    if hasattr(result, 'interrupts') and getattr(result, 'interrupts', None):
        return list(result.interrupts)
    
    # Old LangGraph API: result is a dict with __interrupt__ key
    if isinstance(result, dict):
        legacy = result.get('__interrupt__')
        if legacy:
            return list(legacy)
    
    return []  # No interrupt — graph completed normally


# --- Test Case 1: new API — result object with .interrupts attribute ---
class FakeInterrupt:
    def __init__(self, value):
        self.value = value

class NewAPIResult:
    def __init__(self):
        self.interrupts = [FakeInterrupt({'tool': 'create_issue', 'args': {'subject': 'Login page'}})]
        self.messages = []

result_new = NewAPIResult()
interrupts = _extract_interrupts(result_new)
print('Test 1 — new API:')
print('  Found', len(interrupts), 'interrupt(s)')
print('  Value:', interrupts[0].value)


# --- Test Case 2: old API — result dict with __interrupt__ key ---
result_old = {
    '__interrupt__': [{'value': {'tool': 'update_issue_status', 'args': {'issue_id': 42, 'status_id': 5}}}],
    'messages': []
}
interrupts_old = _extract_interrupts(result_old)
print('\nTest 2 — old API (dict):')
print('  Found', len(interrupts_old), 'interrupt(s)')
print('  Value:', interrupts_old[0])


# --- Test Case 3: no interrupt — normal completion ---
class NormalResult:
    def __init__(self):
        self.messages = [AIMessage(content='Here are your tasks...')]

result_normal = NormalResult()
interrupts_none = _extract_interrupts(result_normal)
print('\nTest 3 — no interrupt:')
print('  Found', len(interrupts_none), 'interrupt(s) → normal flow')


# --- Test Case 4: empty dict ---
result_empty = {}
print('\nTest 4 — empty dict:', _extract_interrupts(result_empty))

Normalizes an interrupt object into a plain Python dict
so it can be serialized to JSON and sent to the frontend.

In [ ]:
def _interrupt_to_payload(interrupt_obj: Any) -> Dict[str, Any]:
    # Case 1: object with a .value attribute (most common in new LangGraph)
    if hasattr(interrupt_obj, 'value'):
        value = interrupt_obj.value
        return value if isinstance(value, dict) else {'value': value}
    
    # Case 2: plain dict (old LangGraph or middleware)
    if isinstance(interrupt_obj, dict):
        value = interrupt_obj.get('value', interrupt_obj)
        return value if isinstance(value, dict) else {'value': value}
    
    # Case 3: anything else — wrap it
    return {'value': interrupt_obj}


class FakeInterrupt:
    def __init__(self, value):
        self.value = value


# --- Test 1: object whose .value is already a dict ---
i1 = FakeInterrupt({'tool': 'create_issue', 'args': {'subject': 'Login page', 'priority_id': 5}})
print('Test 1 — .value is dict:')
print(' ', _interrupt_to_payload(i1))


# --- Test 2: object whose .value is a plain string ---
i2 = FakeInterrupt('Pending action')
print('\nTest 2 — .value is string (wrapped in dict):')
print(' ', _interrupt_to_payload(i2))


# --- Test 3: plain dict with a value key ---
i3 = {'value': {'tool': 'reassign_issue', 'args': {'issue_id': 7, 'assigned_to_id': 3}}}
print('\nTest 3 — plain dict with value key:')
print(' ', _interrupt_to_payload(i3))


# --- Test 4: plain dict without a value key (is itself the payload) ---
i4 = {'tool': 'log_time', 'args': {'issue_id': 5, 'hours': 2.5}}
print('\nTest 4 — plain dict without value key:')
print(' ', _interrupt_to_payload(i4))


# --- Test 5: something completely unexpected ---
i5 = 42
print('\nTest 5 — unexpected type (wrapped):')
print(' ', _interrupt_to_payload(i5))


# --- Show what the frontend receives ---
print('\nWhat the frontend receives (JSON):')
payload = _interrupt_to_payload(i1)
print(json.dumps(payload, indent=2))

Extracts the text of the last message from whatever the graph returned.
Handles both normal completion and post-approval resume results.

In [ ]:
def _extract_last_message_content(result: Any) -> str:
    # After a resume, result might be wrapped in a .value attribute
    payload = result.value if hasattr(result, 'value') else result

    # Get the messages list from whatever payload is
    if isinstance(payload, dict):
        messages = payload.get('messages', [])
    else:
        messages = getattr(payload, 'messages', [])

    if not messages:
        return 'No response message produced.'
    
    last = messages[-1]
    # AIMessage has .content, plain dicts might not
    return getattr(last, 'content', str(last))


# --- Test 1: normal result dict with messages ---
result1 = {
    'messages': [
        AIMessage(content='First response'),
        AIMessage(content='Final answer: here are your tasks...')
    ]
}
print('Test 1 — dict with messages:')
print(' ', _extract_last_message_content(result1))


# --- Test 2: result object with .messages attribute ---
class ResultObj:
    def __init__(self):
        self.messages = [AIMessage(content='Task created successfully.')]

result2 = ResultObj()
print('\nTest 2 — object with .messages:')
print(' ', _extract_last_message_content(result2))


# --- Test 3: result wrapped in .value (post-resume) ---
class ResumeResult:
    def __init__(self):
        self.value = {
            'messages': [AIMessage(content='Issue #47 created successfully.')]
        }

result3 = ResumeResult()
print('\nTest 3 — wrapped in .value (post-resume):')
print(' ', _extract_last_message_content(result3))


# --- Test 4: empty messages list ---
result4 = {'messages': []}
print('\nTest 4 — empty messages:')
print(' ', _extract_last_message_content(result4))


# --- Test 5: messages list with plain dict instead of AIMessage ---
result5 = {'messages': [{'content': 'Plain dict message'}]}
print('\nTest 5 — plain dict in messages (no .content attr):')
print(' ', _extract_last_message_content(result5))

This simulates the complete flow without a real LangGraph agent or Redmine:
1. User sends a message → agent wants to create an issue → interrupt fires
2. Frontend receives requires_human=True with the pending action
3. User approves → graph resumes → issue is created → final response

In [ ]:
# ── Mock objects ───────────────────────────────────────────────────────────────

class MockInterrupt:
    """Simulates a LangGraph interrupt object."""
    def __init__(self, tool_name, args):
        self.value = {
            'action_requests': [{
                'tool': tool_name,
                'args': args
            }]
        }

class MockInterruptedResult:
    """Simulates app.invoke() result when a write tool is intercepted."""
    def __init__(self):
        self.interrupts = [
            MockInterrupt('create_issue', {
                'project_id': 'ai-chatbot-platform',
                'subject': 'Implement login page',
                'priority_id': 5,
                'assigned_to_id': 3
            })
        ]
        self.messages = []

class MockNormalResult:
    """Simulates app.invoke() result after successful resume."""
    def __init__(self):
        self.messages = [
            AIMessage(content='✅ Issue #47 "Implement login page" created successfully in ai-chatbot-platform.')
        ]


# ── Step 1: User sends a message → agent intercepts write tool ─────────────────
print('=' * 60)
print('STEP 1 — User sends message')
print('=' * 60)

request = ChatRequest(
    message='Create a ticket for implementing the login page, assign to Ahmed, high priority',
    thread_id='session-demo'
)
print(f'Message  : {request.message}')
print(f'Thread ID: {request.thread_id}')

# Simulate app.invoke() returning an interrupted result
result = MockInterruptedResult()

interrupts = _extract_interrupts(result)
print(f'\nInterrupts detected: {len(interrupts)}')

if interrupts:
    pending = _interrupt_to_payload(interrupts[0])
    response = ChatResponse(
        response='Action waiting for human confirmation.',
        requires_human=True,
        interrupts=pending
    )
    print('\nAPI Response to frontend:')
    print(json.dumps(response.model_dump(), indent=2))


# ── Step 2: Frontend shows confirmation dialog ─────────────────────────────────
print('\n' + '=' * 60)
print('STEP 2 — Frontend shows confirmation dialog')
print('=' * 60)
action = pending.get('action_requests', [{}])[0]
print(f"Tool    : {action.get('tool')}")
print(f"Args    : {json.dumps(action.get('args', {}), indent=2)}")
print('\nUser sees: "I am about to create this issue — Approve or Reject?"')


# ── Step 3: User approves ──────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('STEP 3 — User approves')
print('=' * 60)

approve_req = ApproveRequest(decision_type='approve')
print(f'Decision: {approve_req.decision_type}')

decision = {'type': approve_req.decision_type}
print(f'Command sent to LangGraph: Command(resume={{"decisions": [{decision}]}})')


# ── Step 4: Graph resumes → tool executes → final response ────────────────────
print('\n' + '=' * 60)
print('STEP 4 — Graph resumes, tool executes, final response')
print('=' * 60)

# Simulate app.invoke(Command(resume=...)) returning a normal result
resume_result = MockNormalResult()
final_text = _extract_last_message_content(resume_result)

final_response = {
    'status': approve_req.decision_type,
    'response': final_text
}
print('Final API response:')
print(json.dumps(final_response, indent=2))

## Reject Flow Simulation

In [ ]:
print('=' * 60)
print('REJECT FLOW')
print('=' * 60)

reject_req = ApproveRequest(decision_type='reject', message='Wrong project selected')
print(f'Decision : {reject_req.decision_type}')
print(f'Message  : {reject_req.message}')

decision = {'type': reject_req.decision_type}
if reject_req.message:
    decision['message'] = reject_req.message

print(f'\nCommand sent to LangGraph: Command(resume={{"decisions": [{decision}]}})')

# After rejection, agent gets the feedback and responds
class MockRejectedResult:
    def __init__(self):
        self.messages = [
            AIMessage(content='Understood. The issue creation was cancelled. Please specify the correct project and I will try again.')
        ]

reject_result = MockRejectedResult()
print('\nFinal response after rejection:')
print(_extract_last_message_content(reject_result))

## Edit Flow Simulation

In [ ]:
print('=' * 60)
print('EDIT FLOW')
print('=' * 60)

edit_req = ApproveRequest(
    decision_type='edit',
    edited_action={
        'name': 'create_issue',
        'args': {
            'project_id': 'e-commerce-platform',   # corrected
            'subject': 'Implement login page',
            'priority_id': 4,                       # changed from 5 to 4
            'assigned_to_id': 3
        }
    }
)

print('Original args : project=ai-chatbot-platform, priority=5')
print('Edited args   :', json.dumps(edit_req.edited_action['args'], indent=2))

decision = {
    'type': 'edit',
    'edited_action': edit_req.edited_action
}
print('\nCommand sent to LangGraph:')
print(json.dumps({'decisions': [decision]}, indent=2))

## Streaming Response — How SSE Works
The `/chat/stream` endpoint yields events one by one using Server-Sent Events.

In [ ]:
# Simulate what chat_stream() yields
def mock_chat_stream(question, thread_id):
    """Simulates the stream of events from chat_stream()."""
    yield {'type': 'routing',  'agent': 'supervisor', 'content': 'Routing to tasks_agent'}
    yield {'type': 'thinking', 'agent': 'tasks_agent', 'content': 'Calling tool: get_projects'}
    yield {'type': 'thinking', 'agent': 'tasks_agent', 'content': 'Tool result received'}
    yield {'type': 'thinking', 'agent': 'tasks_agent', 'content': 'Calling tool: get_issues'}
    yield {'type': 'thinking', 'agent': 'tasks_agent', 'content': 'Tool result received'}
    yield {'type': 'answer',   'agent': 'tasks_agent', 'content': 'Here are the 3 open tasks...'}


# Simulate the SSE event_generator function
def event_generator(question, thread_id):
    for event in mock_chat_stream(question, thread_id):
        yield f"data: {json.dumps(event)}\n\n"
    yield 'data: [DONE]\n\n'


print('SSE stream output (what the frontend receives):')
print('-' * 60)
for chunk in event_generator('List open tasks', 'session-1'):
    print(repr(chunk))  # repr() shows \n characters visibly

print('-' * 60)
print('\nParsed events (what the frontend reads):')
for chunk in event_generator('List open tasks', 'session-1'):
    raw = chunk.strip()
    if raw == 'data: [DONE]':
        print('→ Stream complete')
        break
    data = json.loads(raw.replace('data: ', ''))
    print(f"  [{data['type']:8s}] {data['agent']:12s} → {data['content']}")

## Summary

| Function | Purpose | Essential ? |
|---|---|---|
| `_extract_interrupts` | Finds interrupts in whatever `app.invoke()` returns | Yes — LangGraph API inconsistency |
| `_interrupt_to_payload` | Normalizes interrupt into a plain dict for JSON | Yes — frontend needs clean JSON |
| `_extract_last_message_content` | Gets the final text response from the graph result | Yes — result shape varies |
| `isinstance` | Checks type before accessing type-specific methods | Built-in safety pattern |
| `hasattr` | Checks attribute existence before accessing it | Built-in safety pattern |